In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# === KONFIGURASI PROYEK ===
NIM = 24146088  

# Path lokasi folder dataset yang sudah diekstrak
DATASET_DIR = './flowers'

# Nama kelas sesuai folder dataset
CATEGORIES = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

# Dimensi gambar setelah di-resize (64x64 piksel agar efisien untuk MLP)
IMG_SIZE = 64

data = []
labels = []
original_images = []  # Disimpan untuk visualisasi plot di akhir

print("Memulai proses pemuatan dan preprocessing data...")

for category_idx, category in enumerate(CATEGORIES):
    folder_path = os.path.join(DATASET_DIR, category)
    
    if not os.path.exists(folder_path):
        print(f"Peringatan: Folder {folder_path} tidak ditemukan!")
        continue
        
    print(f"Memproses kelas: {category}...")
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        
        # 1. BACA GAMBAR
        img = cv2.imread(img_path)
        if img is None:
            continue  # Abaikan file jika terdistorsi / bukan gambar
            
        # 2. KONVERSI WARNA BGR ke RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 3. RESIZE GAMBAR
        img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        
        # 4. EKSTRAKSI FITUR (FLATTEN)
        # Mengubah matriks 3D (64, 64, 3) menjadi Vektor 1D (12288,) untuk MLP
        img_flatten = img_resized.flatten()
        
        data.append(img_flatten)
        labels.append(category_idx)
        original_images.append(img_resized)

# Konversi ke NumPy Array
X = np.array(data, dtype='float32')
y = np.array(labels)
original_images = np.array(original_images)

# 5. NORMALISASI PIKSEL (Ubah rentang 0-255 menjadi 0.0 - 1.0)
X = X / 255.0

print(f"\nSelesai! Total gambar yang diproses: {X.shape[0]}")
print(f"Dimensi Vektor Fitur: {X.shape}")

# Pembagian data 80% train dan 20% test
# random_state menggunakan NIM sesuai instruksi soal
X_train, X_test, y_train, y_test, orig_train, orig_test = train_test_split(
    X, 
    y, 
    original_images,
    test_size=0.2, 
    random_state=NIM,
    stratify=y  # Menjaga proporsi tiap kelas seimbang
)

print(f"Jumlah Data Training (80%) : {X_train.shape[0]}")
print(f"Jumlah Data Testing  (20%) : {X_test.shape[0]}")

print("Melatih model MLPClassifier...")

# Inisialisasi Model MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),  # 3 Hidden Layer
    activation='relu',                  # Fungsi aktivasi ReLU
    solver='adam',                      # Optimizer Adam
    max_iter=300,                       # Jumlah iterasi max
    random_state=NIM,                   # NIM sebagai random state
    verbose=True                        # Menampilkan proses training tiap epoch
)

# Proses Training
mlp.fit(X_train, y_train)

print("\nProses Pelatihan Selesai!")

# Prediksi pada Data Testing
y_pred = mlp.predict(X_test)

# Menampilkan Laporan Klasifikasi dengan ketelitian 4 angka desimal
print("="*60)
print("                     CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=CATEGORIES, digits=4))

# Membuat Plot Grid 5x5
plt.figure(figsize=(15, 15))

for row_idx, category in enumerate(CATEGORIES):
    # Cari indeks data testing yang tergolong dalam kelas saat ini
    class_indices = np.where(y_test == row_idx)[0]
    
    # Ambil 5 gambar secara acak dengan seed NIM agar hasil konsisten
    np.random.seed(NIM)
    selected_indices = np.random.choice(class_indices, size=5, replace=False)
    
    for col_idx, img_idx in enumerate(selected_indices):
        plot_position = row_idx * 5 + col_idx + 1
        plt.subplot(5, 5, plot_position)
        
        # Tampilkan Gambar
        plt.imshow(orig_test[img_idx])
        plt.axis('off')
        
        true_label = CATEGORIES[y_test[img_idx]]
        pred_label = CATEGORIES[y_pred[img_idx]]
        
        # Warna teks: Hijau jika prediksi benar, Merah jika salah
        text_color = 'green' if true_label == pred_label else 'red'
        
        plt.title(f"True: {true_label}\nPred: {pred_label}", color=text_color, fontsize=10)

plt.suptitle("Hasil Prediksi 25 Gambar Acak (5 Per Kelas)", fontsize=16, y=0.92)
plt.tight_layout()
plt.show()

